# Multi-Agent Systems

A **multi-agent** system splits work across several specialized agents. A common pattern is a **supervisor** (main agent) that delegates to **subagents**, each focused on a narrow task. The subagents are exposed to the main agent as tools.

In this notebook you'll:
1. Build two specialized subagents (one for square root, one for square).
2. Wrap each subagent as a tool.
3. Create a main agent that decides which subagent to call and route the request.

**Key learnings**
- Specialist agents collaborate via handoffs
- A supervisor/router can delegate subtasks
- Shared or isolated state designs trade off control vs autonomy


### Load environment variables

`load_dotenv()` initializes the environment by loading the `.env` file so the model provider API key is available.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## Creating subagents

Each subagent is a small agent with a single focused tool. Here we define the underlying math tools first, then wrap them in dedicated agents.

### Define the math tools

Two simple tools: `square_root` returns the square root of a number, and `square` returns its square. These are the capabilities each subagent will specialize in.

In [ ]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

### Build the two subagents

`subagent_1` is given only the `square_root` tool, and `subagent_2` only the `square` tool. Each is a standalone agent focused on one job, the building blocks the main agent will delegate to.

In [ ]:
from langchain.agents import create_agent

# create subagents

subagent_1 = create_agent(
    model='gpt-5-nano',
    tools=[square_root]
)

subagent_2 = create_agent(
    model='gpt-5-nano',
    tools=[square]
)

## Calling subagents

To let a main agent use the subagents, we wrap each subagent invocation in a tool. The main agent then treats "call subagent 1" and "call subagent 2" as ordinary tools it can choose between.

### Wrap subagents as tools and create the main agent

`call_subagent_1` and `call_subagent_2` each invoke their subagent with a phrased request and return the subagent's final message. The `main_agent` is created with these two wrapper tools and a system prompt telling it it can delegate square-root and square calculations to the subagents.

In [ ]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model='gpt-5-nano',
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

## Test

Now we exercise the system end to end: ask the main agent a question and watch it route the work to the right subagent.

### Ask the main agent

We ask for the square root of 456. The main agent should recognize this as a square-root task and call `call_subagent_1`, which delegates to `subagent_1` and its `square_root` tool.

In [ ]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

### Inspect the response

`pprint` shows the full message history, including the main agent's delegation to the subagent and the final answer that comes back.

In [ ]:
from pprint import pprint

pprint(response)